# Stage 20A: Same-State Full-SOAP Transmission Test

Stage 19 demonstrated that destroying MoQ's exact coordinate alignment causes the moment-pathway gain $r_t$ to skyrocket, yet actual training parameter updates do not proportionally blow up. This implies the full SOAP map strongly attenuates the geometric perturbation.

In this notebook, we follow a single *natural* training trajectory. At target steps, we perform an explicit same-state counterfactual analysis:
1. We freeze the optimizer state.
2. We compute the parameter update $d_{\text{nat}}$ that natural MoQ produces.
3. We generate 50 magnitude-permuted, $F_{10}$-controlled interventions $g_{\text{int}}$, and compute their corresponding parameter updates $d_{\text{int}}$.
4. We measure the geometric gain ratio $r_{\text{int}}/r_{\text{nat}}$ versus the full update sensitivity $E_{\text{update}} = \|d_{\text{int}}-d_{\text{nat}}\|/\|d_{\text{nat}}\|$.

If $r_{\text{int}}/r_{\text{nat}} \gg 1$ but $E_{\text{update}} \ll r_{\text{int}}/r_{\text{nat}}$, it proves that the full SOAP map shields the parameter updates from the severe coordinate-level sensitivity exposed by the raw denominator.

In [ ]:
import json
import math
import sys
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

sys.path.append('.')
from official_soap import SOAP


In [ ]:
class SOAPStage20A(SOAP):
    def __init__(self, params, lr=1e-3, betas=(0.95, 0.95), shampoo_beta=-1, 
                 eps=1e-8, weight_decay=0.0, precondition_frequency=10, 
                 max_precond_dim=10000, merge_dims=False, precondition_1d=False,
                 normalize_grads=False, data_format='channels_first', correct_bias=True,
                 mu=0.5, damping=1e-3):
        super().__init__(params, lr=lr, betas=betas, shampoo_beta=shampoo_beta, 
                         eps=eps, weight_decay=weight_decay, 
                         precondition_frequency=precondition_frequency,
                         max_precond_dim=max_precond_dim, merge_dims=merge_dims, 
                         precondition_1d=precondition_1d, normalize_grads=normalize_grads, 
                         data_format=data_format, correct_bias=correct_bias)
        self.mu = mu
        self.damping = damping
        self.layer_diagnostics = []
        self.target_steps = {50, 150, 300, 450}
        self.target_layers = {0, 2, 4, 6, 8}
        self.n_perm = 50

    def functional_soap_update(self, raw_gradient, state, group, p):
        b1, b2 = group['betas']
        eps = group['eps']
        
        proj_grad = self.project(raw_gradient.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        m_old = state.get("exp_avg", torch.zeros_like(proj_grad))
        v_old = state.get("exp_avg_sq", torch.zeros_like(proj_grad))
        
        m_new = m_old * b1 + proj_grad * (1.0 - b1)
        v_new = v_old * b2 + proj_grad.square() * (1.0 - b2)
        
        denom = v_new.sqrt() + eps
        
        step_val = state.get('step', 0) + 1
        
        step_size = group['lr']
        if getattr(self, "correct_bias", True):
            bias_correction1 = 1.0 - b1 ** step_val
            bias_correction2 = 1.0 - b2 ** step_val
            step_size = step_size * (math.sqrt(bias_correction2)) / bias_correction1
            
        norm_grad = self.project_back(m_new / denom, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        if group.get("normalize_grads", False):
            norm_grad = norm_grad / (1e-30 + torch.mean(norm_grad**2)**0.5)
            
        update = -step_size * norm_grad
        
        if group.get("weight_decay", 0.0) > 0.0:
            update = update - group['lr'] * group['weight_decay'] * p.data
            
        return update

    @torch.no_grad()
    def step(self, closure=None):
        loss = None if closure is None else closure()
        self.layer_diagnostics = []
        
        current_step = 0
        if len(self.param_groups) > 0 and len(self.param_groups[0]['params']) > 0:
            p_sample = self.param_groups[0]['params'][0]
            if 'step' in self.state[p_sample]:
                current_step = self.state[p_sample]['step'] + 1
        else:
            current_step = 1
            
        idx = 0
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    idx += 1
                    continue
                    
                layer_idx = idx
                state = self.state[p]
                raw_g = p.grad.clone()
                
                if 'moq_initialized' not in state:
                    state['moq_initialized'] = True
                    state['prev_raw_g'] = raw_g.clone()
                    g_moq = raw_g.clone()
                else:
                    if current_step in self.target_steps and layer_idx in self.target_layers and p.dim() > 1 and 'Q' in state and state['Q'] is not None:
                        g_nat = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                        d_nat = self.functional_soap_update(g_nat, state, group, p)
                        norm_d_nat = float(torch.linalg.vector_norm(d_nat))
                        
                        proj_raw = self.project(raw_g.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                        b1, b2 = group["betas"]
                        v_old = state.get("exp_avg_sq")
                        
                        if v_old is not None:
                            v_raw = v_old * b2 + proj_raw.square() * (1.0 - b2)
                            denom_raw = v_raw.sqrt() + group["eps"]
                            D_inv = 1.0 / denom_raw
                            
                            m_old = state.get("exp_avg", torch.zeros_like(proj_raw))
                            m_raw = m_old * b1 + proj_raw * (1.0 - b1)
                            m_raw_norm = float(torch.linalg.vector_norm(m_raw))
                            q_m = m_raw / m_raw_norm if m_raw_norm > 1e-12 else torch.zeros_like(m_raw)
                            z_m = D_inv * q_m
                            norm_z_m = float(torch.linalg.vector_norm(z_m))
                            
                            delta_g_orig = self.mu * (raw_g - state['prev_raw_g'])
                            proj_delta = self.project(delta_g_orig.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                            
                            delta_m = proj_delta * (1.0 - b1)
                            delta_m_norm = float(torch.linalg.vector_norm(delta_m))
                            
                            if delta_m_norm > 1e-12:
                                u_nat = delta_m / delta_m_norm
                                z_nat = D_inv * u_nat
                                norm_z_nat = float(torch.linalg.vector_norm(z_nat))
                                r_nat = norm_z_nat / (norm_z_m + 1e-12)
                                
                                denom_flat = denom_raw.flatten()
                                k10 = max(1, int(0.10 * denom_flat.numel()))
                                _, indices10 = torch.topk(denom_flat, k10, largest=False)
                                
                                tail_mask = torch.zeros_like(denom_flat, dtype=torch.bool)
                                tail_mask[indices10] = True
                                
                                z_delta = D_inv * delta_m
                                z_delta_sq = z_delta.flatten()**2
                                sum_z_delta_sq = float(torch.sum(z_delta_sq)) + 1e-12
                                F10_nat = float(torch.sum(z_delta_sq[indices10])) / sum_z_delta_sq
                                
                                delta_m_flat = delta_m.flatten()
                                d_T_vals = delta_m_flat[tail_mask]
                                d_N_vals = delta_m_flat[~tail_mask]
                                
                                if len(d_T_vals) > 0 and len(d_N_vals) > 0:
                                    r_int_list = []
                                    r_ratio_list = []
                                    norm_d_int_list = []
                                    norm_ratio_list = []
                                    E_update_list = []
                                    cos_list = []
                                    f10_err_list = []
                                    
                                    for _ in range(self.n_perm):
                                        perm_T = torch.randperm(len(d_T_vals), device=d_T_vals.device)
                                        perm_N = torch.randperm(len(d_N_vals), device=d_N_vals.device)
                                        
                                        d_T_perm = d_T_vals[perm_T]
                                        d_N_perm = d_N_vals[perm_N]
                                        
                                        d_perm_flat = torch.zeros_like(delta_m_flat)
                                        d_perm_flat[tail_mask] = d_T_perm
                                        d_perm_flat[~tail_mask] = d_N_perm
                                        d_perm = d_perm_flat.view(delta_m.shape)
                                        
                                        tail_mask_shaped = tail_mask.view(denom_raw.shape)
                                        d_T_full = torch.where(tail_mask_shaped, d_perm, torch.zeros_like(d_perm))
                                        d_N_full = torch.where(~tail_mask_shaped, d_perm, torch.zeros_like(d_perm))
                                        
                                        norm_d_T_sq = float(torch.sum(d_T_full**2))
                                        norm_d_N_sq = float(torch.sum(d_N_full**2))
                                        
                                        E_T = float(torch.sum((D_inv * d_T_full)**2))
                                        E_N = float(torch.sum((D_inv * d_N_full)**2))
                                        
                                        if F10_nat == 0.0:
                                            c_T = 0.0
                                            c_N = 1.0 / math.sqrt(norm_d_N_sq + 1e-12)
                                        elif F10_nat == 1.0:
                                            c_T = 1.0 / math.sqrt(norm_d_T_sq + 1e-12)
                                            c_N = 0.0
                                        else:
                                            ratio = (F10_nat * E_N) / ((1.0 - F10_nat) * E_T + 1e-12)
                                            c_N_sq = 1.0 / (ratio * norm_d_T_sq + norm_d_N_sq + 1e-12)
                                            c_T_sq = c_N_sq * ratio
                                            c_N = math.sqrt(c_N_sq)
                                            c_T = math.sqrt(c_T_sq)
                                            
                                        d_corrected = (c_T * d_T_full + c_N * d_N_full) * delta_m_norm
                                        
                                        u_int = d_corrected / (delta_m_norm + 1e-12)
                                        r_int = float(torch.linalg.vector_norm(D_inv * u_int)) / (norm_z_m + 1e-12)
                                        
                                        actual_z_sq = (D_inv * d_corrected).flatten()**2
                                        actual_F10 = float(torch.sum(actual_z_sq[indices10])) / (float(torch.sum(actual_z_sq)) + 1e-12)
                                        
                                        proj_delta_corrected = d_corrected / (1.0 - b1)
                                        delta_g_corrected = self.project_back(proj_delta_corrected, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                                        g_int = raw_g + delta_g_corrected
                                        
                                        d_int = self.functional_soap_update(g_int, state, group, p)
                                        norm_d_int = float(torch.linalg.vector_norm(d_int))
                                        
                                        E_update = float(torch.linalg.vector_norm(d_int - d_nat)) / (norm_d_nat + 1e-12)
                                        cos_sim = float(torch.sum(d_nat * d_int)) / (norm_d_nat * norm_d_int + 1e-12)
                                        
                                        r_int_list.append(r_int)
                                        r_ratio_list.append(r_int / (r_nat + 1e-12))
                                        norm_d_int_list.append(norm_d_int)
                                        norm_ratio_list.append(norm_d_int / (norm_d_nat + 1e-12))
                                        E_update_list.append(E_update)
                                        cos_list.append(cos_sim)
                                        f10_err_list.append(abs(actual_F10 - F10_nat))
                                        
                                    diag = {
                                        "step": current_step,
                                        "layer": layer_idx,
                                        "r_nat": r_nat,
                                        "F10_nat": F10_nat,
                                        "norm_d_nat": norm_d_nat,
                                        "r_int_median": float(np.median(r_int_list)),
                                        "r_ratio_median": float(np.median(r_ratio_list)),
                                        "r_ratio_q25": float(np.percentile(r_ratio_list, 25)),
                                        "r_ratio_q75": float(np.percentile(r_ratio_list, 75)),
                                        "norm_ratio_median": float(np.median(norm_ratio_list)),
                                        "norm_ratio_q25": float(np.percentile(norm_ratio_list, 25)),
                                        "norm_ratio_q75": float(np.percentile(norm_ratio_list, 75)),
                                        "E_update_median": float(np.median(E_update_list)),
                                        "E_update_q25": float(np.percentile(E_update_list, 25)),
                                        "E_update_q75": float(np.percentile(E_update_list, 75)),
                                        "cos_median": float(np.median(cos_list)),
                                        "max_F10_err": float(np.max(f10_err_list))
                                    }
                                    self.layer_diagnostics.append(diag)
                    
                    # Always apply the natural update
                    g_moq = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                    state['prev_raw_g'].copy_(raw_g)
                
                p.grad.copy_(g_moq)
                idx += 1
                
        super().step()
        return loss


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None
    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule

def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def run_stage20a_audit(K: int=1, steps: int=450, seed: int=0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule_long.pt")
    if schedule_path.exists():
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
        if len(schedule) < steps:
            schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
            torch.save(schedule, schedule_path)
    else:
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    opt = SOAPStage20A(
        model.parameters(), 
        lr=1e-3, 
        betas=(0.95, 0.95), 
        precondition_frequency=10, 
        mu=0.5, 
        damping=1e-3
    )

    history = []
    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        opt.step()
        
        if len(opt.layer_diagnostics) > 0:
            history.extend(opt.layer_diagnostics)
            
        if step % 50 == 0 or step == 1:
            err = relative_l2(model, device)
            print(f"Step {step:3d} | Loss {loss.item():.4e} | RelL2 {err:.4e}")
            if not math.isfinite(loss.item()) or not math.isfinite(err):
                print(f"Divergence detected at step {step}")
                break

    return history


### Stage 20A: Same-State Full-SOAP Transmission Audit

In [ ]:
import matplotlib.pyplot as plt

all_history = {}
for K in [10, 20]:
    print(f"\n{'='*40}\nRunning K={K}...\n{'='*40}")
    history = run_stage20a_audit(K=K, steps=450)
    all_history[K] = history
    with open(f"stage20a_K{K}_soap_moq.json", "w") as f:
        json.dump(history, f, indent=2)

# Main Key Plot
for K, history in all_history.items():
    r_ratio = [math.log10(h["r_ratio_median"]) for h in history]
    e_update = [h["E_update_median"] for h in history]
    plt.figure(figsize=(8, 6))
    plt.scatter(r_ratio, e_update, alpha=0.7)
    plt.xlabel('log10(r_int / r_nat)')
    plt.ylabel('E_update (full SOAP)')
    plt.title(f'K={K}: Geometric Amplification vs Update Sensitivity')
    plt.grid(True)
    plt.savefig(f'stage20a_K{K}_transmission.png')
    plt.close()
